# AURA DIAMOND v0.3 — Evaluation
Analyze training results, compare scenarios, validate thesis.

In [ ]:
!git clone https://github.com/SotoAlt/aura.git /content/aura 2>/dev/null || (cd /content/aura && git pull)
!pip install -q -r /content/aura/world_model/requirements-diamond.txt
import torch; print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
!cd /content/aura && python -m world_model.diamond.eval --checkpoint checkpoints/diamond.ckpt --data /content/video_data --output /content/eval_diamond

In [ ]:
from IPython.display import Image, display
from pathlib import Path
for gif in sorted(Path("/content/eval_diamond").glob("*.gif")):
    print(f"\n--- {gif.stem} ---")
    display(Image(filename=str(gif)))

In [ ]:
import json
with open("/content/eval_diamond/metrics.json") as f:
    m = json.load(f)
print("Success Criteria:")
for k, v in m.get("diamond_criteria", {}).items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")

In [ ]:
import numpy as np
from world_model.diamond.sample import load_model, imagine
from world_model.eval import make_gif

model, cfg = load_model("checkpoints/diamond.ckpt")
device = next(model.parameters()).device

# Custom audio: silence -> bass drop -> full energy
T = 100
ctx = np.zeros((T, 16), dtype=np.float32)
ctx[30:, 12:14] = 0.9  # RMS energy kicks in at frame 30
ctx[30:, 0:2] = 0.8    # bass
ctx[50:, 6:8] = 1.0    # onset burst at frame 50

seed = np.load(sorted(Path("/content/video_data").glob("episode_*.npz"))[0])["image"][:4]
frames = imagine(model, seed, ctx, T, device)
make_gif(frames, "/content/eval_diamond/custom_bass_drop.gif", fps=10, scale=4)
from IPython.display import Image; display(Image(filename="/content/eval_diamond/custom_bass_drop.gif"))

In [ ]:
import shutil
shutil.make_archive("/content/eval_diamond_results", "zip", "/content/eval_diamond")
from google.colab import files
files.download("/content/eval_diamond_results.zip")